## Misclassified Crimes in LAPD Data (Los Angeles Times)

- [Times Investigation: LAPD misclassified nearly 1,200 violent crimes as minor offenses](https://www.latimes.com/local/la-me-crimestats-lapd-20140810-story.html)
- [LAPD underreported serious assaults, skewing crime stats for 8 years](https://www.latimes.com/local/cityhall/la-me-crime-stats-20151015-story.html)
- [How we reported this story](https://www.latimes.com/local/cityhall/la-me-crime-stats-side-20151015-story.html)

## Definitions

>**Aggravated Assault:** An unlawful attack by one person upon another for the purpose of inflicting severe or aggravated bodily injury. This type of assault usually is accompanied by the use of a weapon or by means likely to produce death or great bodily harm.


>**Other Assault:** Simple, Not Aggravated. Includes all assaults which do not involve the use of a firearm, knife, cutting instrument, or other dangerous weapon and in which the victim did not sustain serious or aggravated injuries. 

## Our Data Sample

The dataset has hundreds of thousands of rows, but we will sample 100 from them for now: https://docs.google.com/spreadsheets/d/1LZ72b3cgVi7mhryMiromE3eT86DSnfna1cXjX-jLvGk/edit#gid=0

## Setup

Run the cell below to install the dependencies for this notebook.

In [ ]:
# %pip install -r requirements.txt

## Load the data

In [ ]:
%matplotlib inline
import csv, requests, os
import pandas as pd
import numpy as np

In [ ]:
def make_regular_gsheet_url(doc_id, sheet_id):
    return f"https://docs.google.com/spreadsheets/d/{doc_id}/edit#gid={sheet_id}"

def make_csv_gsheet_url(doc_id, sheet_id):
    return f"https://docs.google.com/spreadsheets/d/{doc_id}/export?format=csv&id={doc_id}&gid={sheet_id}"

GOOGLE_SHEET_ID = '1LZ72b3cgVi7mhryMiromE3eT86DSnfna1cXjX-jLvGk'
google_sheet_url = make_regular_gsheet_url(GOOGLE_SHEET_ID, "0")
print("Querying Doc:", google_sheet_url)

google_sheet_csv_url = make_csv_gsheet_url(GOOGLE_SHEET_ID, "0")
response = requests.get(google_sheet_csv_url)
reader = csv.reader(response.text.splitlines())
header = next(reader)
df = pd.DataFrame(list(reader), columns=header)


# You are the classifier 👈


Based on the definitions provided, categorize the data you have been assigned as `Other Assault` or `Aggrevated Assault`.

In [ ]:
google_sheet_url

## LLM as the classifier 🤖

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
openrouter_key = os.getenv("OPENROUTER_API_KEY")

In [ ]:
import diskcache
cache = diskcache.Cache('./cache')  # stores in ./cache folder

This is a **pydantic model**. It defines what format I want the output to come back in. It's for an LLM feature called "Structured Outputs", but also works with other LLM tools. This gets the LLM to always return structured json data, which gets read into python as a class.

In [ ]:
from pydantic import BaseModel

# This is a pydantic model. It defines what format I want the output to come back in
# It's for an OpenAI feature called "Structured Output", but also works with other LLM tools
class Classification(BaseModel):
    llm: bool
    reason: str

#### Write Your System Prompt 👈

Write a system prompt that tells the model to classify each incident description into exactly one of two categories: `Aggravated Assault` or `Other Assault`

In [ ]:
SYSTEM_PROMPT = ""

In [ ]:
from openai import OpenAI
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=openrouter_key,
)

@cache.memoize() # This is the diskcache! Now I will never hit the API twice with the same request!
def ask_model_to_classify(text_description, model):
  response = client.beta.chat.completions.parse(
    model=model,
    messages=[
      {
        "role": "system",
        "content": SYSTEM_PROMPT
      },
      {
        "role": "user",
        "content": text_description
      },
    ],
    response_format=Classification,
    temperature=0
  )

  return response.choices[0].message.content

In [ ]:
import json
from tqdm.notebook import tqdm
tqdm.pandas()

df[MODEL] = df['description'].progress_apply(ask_model_to_classify, model=MODEL)
df['llm'] = df[MODEL].apply(lambda x: json.loads(x)['llm'])
df['llm'] = df['llm'].replace({True: 'Aggravated Assault', False: 'Other Assault'})
df['reason'] = df[MODEL].apply(lambda x: json.loads(x)['reason'])

# delete model
del df[MODEL]

## Calculate precision and recall vs golden

In [ ]:
pd.crosstab(df['golden'], df['llm'])

In [ ]:
# use sklearn to calculate precision, recall, f1 and accuracy
from sklearn.metrics import classification_report
print(classification_report(df['golden'], df['llm']))

## Inspect false positives and false negatives

In [ ]:
# Define positive class as "Aggravated Assault"
false_positives = df[
    (df['llm'] == 'Aggravated Assault')
    & (df['golden'] != 'Aggravated Assault')
]

false_negatives = df[
    (df['llm'] != 'Aggravated Assault')
    & (df['golden'] == 'Aggravated Assault')
]

cols_to_show = [
    'description',
    'golden',
    'llm',
    'reason',
]

# Show full text in dataframe cells
pd.set_option('display.max_colwidth', None)

In [ ]:
print(f"False positives: {len(false_positives)}")
false_positives[cols_to_show]

In [ ]:
print(f"False negatives: {len(false_negatives)}")
false_negatives[cols_to_show]